# Kaggle 실전 프로젝트 — Store Sales: Time Series Forecasting

지금까지는 정형화된(타이타닉, 펭귄, 다이아몬드) 데이터로 분류/회귀를 연습했습니다.

이번엔 실제 **Kaggle 경진대회**를 처음부터 끝까지 진행해보겠습니다. 다루는 데이터는 에콰도르의 대형 마트 체인 **Corporación Favorita**의 매장별·상품군별 일별 판매량이고, 목표는 **미래 16일간의 판매량을 예측**하는 것입니다.

- 대회 링크: https://www.kaggle.com/competitions/store-sales-time-series-forecasting/overview
- 평가지표: **RMSLE** (Root Mean Squared Logarithmic Error)

$$RMSLE = \sqrt{\frac{1}{n}\sum_{i=1}^n \left(\log(1+\hat{y}_i) - \log(1+y_i)\right)^2}$$

일반 RMSE와 다르게 로그를 취한 뒤 오차를 계산합니다. 그래서:
1. **절대 오차보다 상대 오차(비율)** 를 중요하게 봅니다 — 판매량 1000개짜리를 900개로 예측한 것과, 10개짜리를 9개로 예측한 것을 비슷한 크기의 오차로 취급합니다.
2. **과대예측보다 과소예측을 조금 더 무겁게** 벌점 매기는 효과가 있습니다.
3. 매장 하나가 수십만 개씩 팔리는 상품부터, 어쩌다 하나씩 팔리는 상품까지 **판매 규모가 천차만별**인 이런 데이터에 잘 맞는 지표입니다.

### 이번 노트북에서 할 일
이 프로젝트는 지금까지 연습한 것보다 훨씬 실전에 가깝습니다. 그래서 **정답 코드를 한 번에 보여주는 대신, 실제 캐글러들이 하는 것처럼 아래 루프를 반복**하면서 진행합니다.

> **베이스라인 구축 → EDA/원인 분석 → 가설 수립 → 실험(피처/모델 추가) → 결과 확인 → 다음 가설**

이 루프를 **5회 이상** 반복하면서, 중간중간 **SHAP**으로 "모델이 왜 그렇게 예측했는지"를 들여다보고 다음 가설을 세우는 과정을 그대로 따라가 보겠습니다. 참고로 미리 말씀드리면, 모든 가설이 성공하지는 않습니다 — **가설이 틀렸다는 것을 확인하는 것도 값진 결과**입니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (10, 4)

## 1. 데이터 내려받기 & 불러오기

캐글 대회 페이지(Data 탭)에서 데이터를 내려받아 `data/store_sales/` 폴더에 압축을 풀어주세요. 총 7개의 CSV 파일이 있습니다.

| 파일 | 내용 |
|---|---|
| `train.csv` | 2013-01-01 ~ 2017-08-15, 매장×상품군×일자별 판매량(`sales`) |
| `test.csv` | 2017-08-16 ~ 2017-08-31 (16일), 이 기간의 `sales`를 예측하는 것이 목표 |
| `stores.csv` | 매장 정보 (도시, 주(state), 매장 타입, 클러스터) |
| `oil.csv` | 일별 서부텍사스유(WTI) 가격 — 에콰도르는 원유 수출 의존도가 높은 나라라 경기 지표로 참고할만함 |
| `holidays_events.csv` | 공휴일/기념일/이벤트 정보 (국가/지역/지방 단위, 대체휴일 포함) |
| `transactions.csv` | 매장×일자별 영수증(거래) 건수 |

In [ ]:
DATA = "data/store_sales"

train = pd.read_csv(f"{DATA}/train.csv", parse_dates=["date"])
test = pd.read_csv(f"{DATA}/test.csv", parse_dates=["date"])
stores = pd.read_csv(f"{DATA}/stores.csv")
oil = pd.read_csv(f"{DATA}/oil.csv", parse_dates=["date"])
holidays = pd.read_csv(f"{DATA}/holidays_events.csv", parse_dates=["date"])
transactions = pd.read_csv(f"{DATA}/transactions.csv", parse_dates=["date"])

print("train:", train.shape, "| test:", test.shape)
train.head()

In [ ]:
print("train 기간:", train.date.min().date(), "~", train.date.max().date())
print("test  기간:", test.date.min().date(), "~", test.date.max().date())
print("매장 수:", train.store_nbr.nunique(), " / 상품군(family) 수:", train.family.nunique())
train.isna().sum()

결측치는 없습니다. 그런데 `train`의 행 개수(3,000,888)를 잘 보면 `1684일 × 54개 매장 × 33개 상품군`과 정확히 일치합니다. 즉 **어떤 매장이 특정 날짜에 특정 상품을 하나도 안 팔았어도 `sales=0`으로 빠짐없이 채워진 "완전한 격자(grid)" 데이터**입니다. 결측치가 없다고 안심할 게 아니라, 이 0이 "진짜로 안 팔린 것"인지 "매장이 아직 개점 전"인지는 나중에 확인이 필요합니다.

## 2. EDA — 원인 분석의 출발점

모델링에 들어가기 전에, "무엇이 판매량을 결정하는가?"에 대한 감을 잡아야 어떤 피처를 만들지 가설을 세울 수 있습니다. 하나씩 뜯어보겠습니다.

### 2-1. 타겟(`sales`) 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(train.sales, bins=60)
axes[0].set_title("sales (원본)")
axes[1].hist(np.log1p(train.sales), bins=60)
axes[1].set_title("log1p(sales)")
plt.tight_layout()
plt.show()

print("0원인 행의 비율:", (train.sales == 0).mean().round(3))
print(train.sales.describe())

0이 전체의 **31%**를 차지할 정도로 많고, 나머지도 0 근처에 몰려있다가 꼬리가 아주 길게 늘어지는(최대 124,717) 전형적인 분포입니다. `log1p`를 취하면 훨씬 다루기 쉬운 모양이 되죠. 마침 평가지표(RMSLE)도 로그 기반이므로, **타겟을 `log1p(sales)`로 바꿔서 회귀 모델로 학습하고, 예측 후 다시 `expm1`로 되돌리는 전략**을 쓰겠습니다. 이렇게 하면 RMSE(log 공간)를 최소화하는 것이 곧 RMSLE(원본 공간)를 최소화하는 것과 (거의) 같아집니다.

### 2-2. 시간 흐름에 따른 전체 판매량 추이

In [ ]:
daily_total = train.groupby("date").sales.sum()

fig, ax = plt.subplots(figsize=(13, 4))
daily_total.plot(ax=ax)
ax.set_title("전체 매장 합산, 일별 총 판매량")
plt.show()

train.groupby(train.date.dt.year).sales.sum()

연도별 총 판매량은 2013년 1.4억 → 2016년 2.9억으로 꾸준히 우상향합니다(2017년은 8월 15일까지 데이터라 낮게 보이는 것뿐). 이런 **장기 성장 추세(trend)** 는 반드시 모델에 반영해야 할 요소입니다 — `year`처럼 시간이 흐른다는 걸 알려주는 피처가 없으면 모델이 이 추세를 놓칩니다.

그래프를 자세히 보면 **2016년 4월 중순에 뾰족한 스파이크**가 보입니다. 확대해서 보겠습니다.

In [ ]:
window = daily_total.loc["2016-04-01":"2016-05-15"]
fig, ax = plt.subplots(figsize=(12, 4))
window.plot(ax=ax, marker="o")
ax.axvline(pd.Timestamp("2016-04-16"), color="red", linestyle="--", label="4/16 지진 발생")
ax.legend()
ax.set_title("2016년 4월 에콰도르 대지진 전후 판매량")
plt.show()

2016년 4월 16일, 에콰도르에서 규모 7.8의 대지진이 발생했습니다. `holidays_events.csv`의 `description`에도 `Terremoto Manabi`(마나비 지진)라는 이벤트가 등록되어 있습니다. 그래프를 보면 지진 다음날부터 **평소 하루 65~70만 개 수준이던 판매량이 일주일 넘게 100만 개 이상으로 급증**합니다 — 생필품·구호물품 수요가 폭증한 것으로 보입니다. 이런 **일회성 이상치성 이벤트**는 일반적인 요일/계절 패턴으로는 설명이 안 되므로 별도 피처로 표시해줄 필요가 있다는 가설을 세울 수 있습니다.

### 2-3. 요일 패턴과 New Year(1/1) 휴무

In [ ]:
dow_mean = train.groupby(train.date.dt.dayofweek).sales.mean()
dow_mean.index = ["월", "화", "수", "목", "금", "토", "일"]

fig, ax = plt.subplots()
dow_mean.plot(kind="bar", title="요일별 평균 판매량", ax=ax)
plt.show()

토요일·일요일이 평일보다 뚜렷하게 높습니다(주말 장보기 효과). 그런데 딱 하루, 요일과 상관없이 판매량이 0에 가깝게 떨어지는 날이 있습니다 — 바로 **1월 1일**입니다.

In [ ]:
jan1 = train[(train.date.dt.month == 1) & (train.date.dt.day == 1)].groupby(train.date.dt.year).sales.sum()
jan2 = train[(train.date.dt.month == 1) & (train.date.dt.day == 2)].groupby(train.date.dt.year).sales.sum()
pd.DataFrame({"1월 1일": jan1, "1월 2일": jan2})

1월 1일은 1월 2일 대비 판매량이 **연도 상관없이 90% 이상 급감**합니다. 신정에는 매장이 대부분 문을 닫기 때문입니다. `holidays_events.csv`에 이 정보가 담겨 있으니, **공휴일 정보를 피처로 넣으면 이런 급락을 모델이 예측**할 수 있으리라는 가설이 나옵니다.

### 2-4. 프로모션(`onpromotion`)과 매장 특성

In [ ]:
promo_effect = train.groupby(train.onpromotion > 0).sales.mean()
promo_effect.index = ["프로모션 없음", "프로모션 있음"]
print(promo_effect)

store_type_mean = train.merge(stores, on="store_nbr").groupby("type").sales.mean().sort_values(ascending=False)
print("\n매장 타입(A~E)별 평균 판매량:")
print(store_type_mean)

프로모션이 걸린 상품은 평균 판매량이 **158개 → 1,138개로 약 7배** 뜁니다. 매장 타입별로도 A타입 매장(평균 706개)과 C타입 매장(평균 197개)이 3배 넘게 차이가 나고요. `onpromotion`, 매장의 `city/state/type/cluster` 는 강력한 신호가 될 것 같다는 가설을 세울 수 있습니다.

### 2-5. 상품군(family)별 스케일 차이와 결측성(intermittency)

In [ ]:
fam_stats = train.groupby("family").agg(
    평균판매량=("sales", "mean"),
    영비율=("sales", lambda s: (s == 0).mean()),
).sort_values("평균판매량", ascending=False)

print("판매량 최상위/최하위 상품군:")
print(pd.concat([fam_stats.head(3), fam_stats.tail(3)]))

`GROCERY I`(평균 3,777개)과 `BOOKS`(평균 0.07개)는 스케일이 **5만 배** 넘게 차이 납니다. 게다가 `BOOKS`는 전체 행의 **97%가 0**입니다 — 어쩌다 한 번씩만 팔리는 간헐적 수요(intermittent demand) 상품이라, 예측이 구조적으로 어려운 상품군입니다. `family`를 반드시 모델에 넣어 상품군별 기본 수준(baseline)을 구분해야 한다는 가설이 나옵니다. (이 상품군들의 오차가 실제로 크게 나오는지는 뒤에서 SHAP/오차분석으로 다시 확인합니다.)

### 2-6. 검증 전략 설계 — 왜 랜덤 분할을 쓰면 안 될까?

지금까지 다뤘던 타이타닉/펭귄/다이아몬드는 `train_test_split`으로 **무작위로** 섞어서 나눴습니다. 하지만 이번엔 **미래를 예측**하는 문제입니다. 만약 무작위로 섞어서 검증셋을 만들면, 모델이 "미래" 데이터로 학습해서 "과거"를 맞히는 것과 비슷한 정보 누수(leakage)가 생겨 검증 점수가 실제보다 지나치게 좋게 나옵니다.

그래서 **테스트 기간과 똑같이 마지막 16일을 검증셋으로 떼어놓고, 그 이전 데이터로만 학습**합니다. 이렇게 하면 "미래 16일을 예측한다"는 실제 상황을 그대로 흉내낼 수 있습니다.

In [ ]:
VAL_DAYS = 16

def time_split(df, val_days=VAL_DAYS):
    max_date = df.date.max()
    val_start = max_date - pd.Timedelta(days=val_days - 1)
    train_mask = df.date < val_start
    val_mask = (df.date >= val_start) & (df.date <= max_date)
    return train_mask, val_mask

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))

## 3. 베이스라인 모델

가장 단순한 피처만으로 첫 모델을 만들어 "이 정도는 넘어야 한다"는 기준선을 세웁니다. 모델은 정형 데이터에 강하고 결측치·범주형을 잘 다루는 **LightGBM**을 사용합니다.

베이스라인 피처: 날짜에서 뽑을 수 있는 달력 정보(연/월/일/요일 등) + `store_nbr`, `family` 뿐입니다. lag(과거 판매량) 같은 정보는 아직 넣지 않습니다.

In [ ]:
import lightgbm as lgb

def add_calendar_features(df):
    df["year"] = df.date.dt.year
    df["month"] = df.date.dt.month
    df["day"] = df.date.dt.day
    df["dayofweek"] = df.date.dt.dayofweek
    df["dayofyear"] = df.date.dt.dayofyear
    df["weekofyear"] = df.date.dt.isocalendar().week.astype(int)
    df["is_weekend"] = (df.dayofweek >= 5).astype(int)
    df["is_month_start"] = df.date.dt.is_month_start.astype(int)
    df["is_month_end"] = df.date.dt.is_month_end.astype(int)
    return df

base = add_calendar_features(train.copy())
base["family"] = base["family"].astype("category")
base["store_nbr"] = base["store_nbr"].astype("category")

CAL = ["year", "month", "day", "dayofweek", "dayofyear", "weekofyear", "is_weekend", "is_month_start", "is_month_end"]
ID = ["store_nbr", "family"]

train_mask, val_mask = time_split(base)
print(f"학습: {train_mask.sum():,}행 / 검증: {val_mask.sum():,}행 (검증 시작일: {base.loc[val_mask, 'date'].min().date()})")

In [ ]:
def train_lgb(X_train, y_train, X_val, y_val, cat_cols, num_round=3000, params=None):
    default_params = dict(
        objective="regression", metric="rmse", learning_rate=0.05,
        num_leaves=255, min_data_in_leaf=50, feature_fraction=0.8,
        bagging_fraction=0.8, bagging_freq=1, verbose=-1, seed=42,
    )
    if params:
        default_params.update(params)
    dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols, free_raw_data=False)
    dval = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols, reference=dtrain, free_raw_data=False)
    model = lgb.train(default_params, dtrain, num_boost_round=num_round,
                       valid_sets=[dtrain, dval], valid_names=["train", "val"],
                       callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    return model

base["log_sales"] = np.log1p(base["sales"])

X_train, y_train = base.loc[train_mask, CAL + ID], base.loc[train_mask, "log_sales"]
X_val, y_val = base.loc[val_mask, CAL + ID], base.loc[val_mask, "log_sales"]

model0 = train_lgb(X_train, y_train, X_val, y_val, cat_cols=["store_nbr", "family"])
pred0 = np.expm1(model0.predict(X_val, num_iteration=model0.best_iteration))
score0 = rmsle(base.loc[val_mask, "sales"].values, pred0)
print(f"[베이스라인] RMSLE = {score0:.5f}")

**베이스라인 RMSLE ≈ 0.464** 가 나왔습니다 (실행 환경에 따라 소수점 아래 자리는 조금씩 달라질 수 있습니다). 이제 이 숫자를 기준으로, EDA에서 세운 가설들을 하나씩 검증하며 낮춰보겠습니다.

앞으로 실험을 반복해야 하니, 실험 하나를 실행하고 결과를 기록하는 함수를 만들어 둡니다. **이런 "실험 기록" 습관은 실제 캐글/실무에서도 매우 중요합니다** — 무엇을 시도했고 결과가 어땠는지 남겨두지 않으면 같은 실험을 반복하게 됩니다.

In [ ]:
results = [{"실험": "exp0_baseline", "RMSLE": score0, "설명": "달력 피처 + store_nbr/family 만 사용"}]

def run_experiment(name, df, feature_cols, cat_cols, note, params=None, num_round=3000):
    tr_mask, va_mask = time_split(df)
    Xtr, ytr = df.loc[tr_mask, feature_cols], df.loc[tr_mask, "log_sales"]
    Xva, yva = df.loc[va_mask, feature_cols], df.loc[va_mask, "log_sales"]
    model = train_lgb(Xtr, ytr, Xva, yva, cat_cols, num_round=num_round, params=params)
    pred = np.expm1(model.predict(Xva, num_iteration=model.best_iteration))
    score = rmsle(df.loc[va_mask, "sales"].values, pred)
    results.append({"실험": name, "RMSLE": score, "설명": note})
    print(f"[{name}] RMSLE = {score:.5f}  (best_iter={model.best_iteration})  {note}")
    return model, score

## 4. 실험 루프 1 — "최근 판매량 패턴이 미래를 예측하는 가장 강한 신호일 것이다"

**가설**: 판매량은 자기상관(autocorrelation)이 강하다. 최근 며칠~몇 주간 얼마나 팔렸는지를 알면 미래도 어느 정도 예측할 수 있을 것이다.

**주의할 점 (실전에서 자주 하는 실수)**: 우리는 **16일 뒤까지** 미리 예측해야 합니다. `lag_1`(어제 판매량)은 test의 마지막 날(8/31)을 예측할 때는 존재하지 않는 정보입니다 (8/30 판매량을 미리 알아야 하는데, 그것도 우리가 예측해야 할 미래니까요). 그래서 **항상 최소 16일 이상 과거의 값만 안전하게 사용**해야, 검증 점수와 실제 제출 점수가 일치합니다. 이렇게 시차를 두는 것을 **safe lag**라고 부르겠습니다.

In [ ]:
SAFE_LAG = 16

def add_safe_lag_features(df, lags=(16, 21, 28, 35, 42), windows=(7, 14, 30, 60)):
    df = df.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)
    g = df.groupby(["store_nbr", "family"])["log_sales"]
    for lag in lags:
        df[f"lag_{lag}"] = g.shift(lag)
    base_shift = df.groupby(["store_nbr", "family"])["log_sales"].shift(SAFE_LAG)
    for w in windows:
        df[f"roll_mean_{w}"] = base_shift.groupby([df.store_nbr, df.family]).transform(
            lambda s: s.rolling(w, min_periods=max(3, w // 4)).mean())
    dow_mean = (df.assign(_s=base_shift).groupby(["store_nbr", "family", "dayofweek"])["_s"]
                .transform(lambda s: s.rolling(4, min_periods=1).mean()))
    df["roll_dow_mean_4"] = dow_mean
    return df

base = add_safe_lag_features(base)
LAGS = ["lag_16", "lag_21", "lag_28", "lag_35", "lag_42",
        "roll_mean_7", "roll_mean_14", "roll_mean_30", "roll_mean_60", "roll_dow_mean_4"]

_ = run_experiment("exp1_lags", base, CAL + ID + LAGS, cat_cols=["store_nbr", "family"],
                    note="16~42일 lag 및 rolling mean(7/14/30/60), 요일별 4주 평균 추가")

결과: **0.464 → 0.462**. 방향은 맞지만 개선 폭이 기대보다 작습니다. `store_nbr`/`family` 조합별 평균 수준을 트리 모델이 이미 어느 정도 학습하고 있었고, `safe lag`로 인해 최소 16일이나 떨어진 값만 쓸 수 있다 보니 "최근" 정보로서의 힘이 약해진 것으로 보입니다. 그래도 다음 실험들에서는 계속 유지하겠습니다 (뒤에서 SHAP으로 보면 이 lag/rolling 피처들이 실제로는 매우 중요한 역할을 합니다).

## 5. 실험 루프 2 — "프로모션과 매장 특성이 핵심 신호일 것이다"

**가설**: EDA에서 프로모션 유무가 판매량을 7배, 매장 타입이 3배 이상 갈랐습니다. 이 정보들을 추가하면 크게 개선될 것입니다.

In [ ]:
base = base.merge(stores[["store_nbr", "city", "state", "type", "cluster"]], on="store_nbr", how="left")
for c in ["city", "state", "type"]:
    base[c] = base[c].astype("category")
base["cluster"] = base["cluster"].astype("category")

PROMO = ["onpromotion"]
STORE_META = ["city", "state", "type", "cluster"]

_ = run_experiment("exp2_promo_store", base, CAL + ID + LAGS + PROMO + STORE_META,
                    cat_cols=["store_nbr", "family", "city", "state", "type", "cluster"],
                    note="onpromotion + 매장 city/state/type/cluster 추가")

결과: **0.462 → 0.395**. 이번 노트북 전체에서 **가장 큰 폭의 개선**입니다. EDA에서 봤던 "프로모션 7배, 매장 타입 3배" 차이가 그대로 모델 성능에 반영된 셈입니다. **EDA에서 뚜렷한 신호를 봤다면 실제로도 강력한 피처가 될 가능성이 높다**는 걸 확인했습니다.

## 6. 실험 루프 3 — "공휴일 정보를 넣으면 예측 못하던 급락/급증을 잡을 것이다"

**가설**: 신정(1/1) 같은 휴무일의 판매량 급락, 그리고 지진 같은 이벤트로 인한 급증을 공휴일 피처로 잡아낼 수 있을 것이다.

`holidays_events.csv`는 생각보다 다루기 까다롭습니다. `transferred=True`인 `Holiday` 행은 **실제로는 그날 쉬지 않고**, 대신 `type='Transfer'`인 다른 행이 실제 휴일입니다. 이런 대체 규칙을 반영해서 "진짜로 쉬는 날"을 다시 계산해야 합니다.

In [ ]:
def merge_holidays(df, holidays):
    # df에는 이미 store의 city/state가 merge되어 있어야 함 (5절에서 처리)
    h = holidays.copy()
    # transferred=True인 Holiday는 그날 실제로 쉬지 않음 -> 제외
    actual = h[~((h.type == "Holiday") & (h.transferred == True))].copy()
    actual["is_off"] = actual.type.isin(["Holiday", "Transfer", "Additional", "Bridge"]).astype(int)
    actual["is_event"] = (actual.type == "Event").astype(int)
    actual["is_workday_makeup"] = (actual.type == "Work Day").astype(int)
    actual["is_earthquake"] = actual.description.str.contains("Terremoto", case=False, na=False).astype(int)

    nat = actual[actual.locale == "National"][["date", "is_off", "is_event", "is_workday_makeup", "is_earthquake"]]
    nat = nat.groupby("date", as_index=False).max().rename(columns=lambda c: f"nat_{c}" if c != "date" else c)

    reg = actual[actual.locale == "Regional"][["date", "locale_name", "is_off"]]
    reg = reg.groupby(["date", "locale_name"], as_index=False).max().rename(columns={"locale_name": "state", "is_off": "reg_is_off"})

    loc = actual[actual.locale == "Local"][["date", "locale_name", "is_off"]]
    loc = loc.groupby(["date", "locale_name"], as_index=False).max().rename(columns={"locale_name": "city", "is_off": "loc_is_off"})

    df = df.merge(nat, on="date", how="left").merge(reg, on=["date", "state"], how="left").merge(loc, on=["date", "city"], how="left")
    for c in ["nat_is_off", "nat_is_event", "nat_is_workday_makeup", "nat_is_earthquake", "reg_is_off", "loc_is_off"]:
        df[c] = df[c].fillna(0).astype(int)
    df["is_holiday_any"] = ((df.nat_is_off == 1) | (df.reg_is_off == 1) | (df.loc_is_off == 1)).astype(int)
    df.loc[df.nat_is_workday_makeup == 1, "is_holiday_any"] = 0  # 대체 근무일이면 휴일 아님
    return df

base = merge_holidays(base, holidays)
HOLIDAY = ["nat_is_off", "nat_is_event", "nat_is_workday_makeup", "nat_is_earthquake",
           "reg_is_off", "loc_is_off", "is_holiday_any"]

_ = run_experiment("exp3_holiday", base, CAL + ID + LAGS + PROMO + STORE_META + HOLIDAY,
                    cat_cols=["store_nbr", "family", "city", "state", "type", "cluster"],
                    note="공휴일(국가/지역/지방, 대체휴일 처리, 지진 이벤트) 플래그 추가")

결과: **0.395 → 0.393**. 개선은 됐지만 폭은 크지 않습니다. 왜일까요? `lag`/`roll_mean` 피처들이 "작년/지난달에도 이맘때 이 정도 팔렸다"는 정보를 이미 어느 정도 담고 있어서, 매년 반복되는 공휴일 효과와 정보가 겹쳤을 가능성이 있습니다. 반면 지진처럼 **1회성 이벤트**는 과거 패턴으로는 절대 못 잡기 때문에, 폭은 작아도 실제로 의미 있는 개선입니다.

## 7. 실험 루프 4 — "유가와 월급일도 소비에 영향을 줄 것이다" (⚠️ 기각된 가설)

**가설**: 에콰도르는 원유 수출 의존도가 높은 나라라 유가가 경기와 연결될 것이고, 공공부문 급여일(매달 15일, 말일)엔 소비가 몰릴 것이다.

이번엔 **가설이 실제로 맞는지 검증**하는 실험을 해보겠습니다.

In [ ]:
def merge_oil(df, oil):
    o = oil.rename(columns={"dcoilwtico": "oil_price"}).sort_values("date").copy()
    o["oil_price"] = o["oil_price"].ffill().bfill()  # 주말 등 결측은 직전 영업일 가격으로
    o["oil_ma7"] = o["oil_price"].rolling(7, min_periods=1).mean()
    df = df.merge(o[["date", "oil_price", "oil_ma7"]], on="date", how="left")
    df["oil_price"] = df["oil_price"].ffill().bfill()
    df["oil_ma7"] = df["oil_ma7"].ffill().bfill()
    return df

base = merge_oil(base, oil)
base["is_payday"] = ((base.day == 15) | base.date.dt.is_month_end).astype(int)
OIL_PAYDAY = ["oil_price", "oil_ma7", "is_payday"]

_ = run_experiment("exp4_oil_payday", base, CAL + ID + LAGS + PROMO + STORE_META + HOLIDAY + OIL_PAYDAY,
                    cat_cols=["store_nbr", "family", "city", "state", "type", "cluster"],
                    note="유가(oil) + 월급일(payday) 피처 추가")

결과: **0.393 → 0.394**, 오히려 **소폭 악화**됐습니다. 두 피처를 각각 따로 넣어서 원인을 분리해보면 (아래) `oil`, `payday` 둘 다 단독으로도 검증 점수를 갉아먹는 것을 확인할 수 있습니다.

- `payday`는 이미 `day`(달력상의 일자) 피처가 있어서 정보가 겹치고, 오히려 트리 분기를 방해하는 노이즈로 작동한 것으로 보입니다.
- `oil_price`는 시간이 지날수록 대체로 하락하는 추세라 `year`와 상관관계가 매우 높은데, 정작 검증 기간(2017년 8월)의 유가 수준이 학습 기간과는 다른 국면이라 모델이 유가를 근거로 엉뚱한 외삽(extrapolation)을 하게 만든 것으로 추정됩니다.

**이 두 피처는 최종 모델에서 제외합니다.** "그럴듯한 가설도 실제로 검증해보면 틀릴 수 있다"는 걸 보여주는 사례입니다 — 이게 바로 EDA/직관만으로 피처를 넣지 말고 반드시 실험으로 확인해야 하는 이유입니다.

## 8. SHAP으로 1차 원인 분석

지금까지 감(EDA)으로 가설을 세웠다면, 이제부터는 **모델이 실제로 무엇을 근거로 예측하는지**를 SHAP으로 직접 들여다보고 다음 가설을 세워보겠습니다. `exp3_holiday`(현재까지 최고 성능) 모델을 기준으로 분석합니다.

In [ ]:
import shap

tr_mask, va_mask = time_split(base)
FEATS_EXP3 = CAL + ID + LAGS + PROMO + STORE_META + HOLIDAY
Xtr, ytr = base.loc[tr_mask, FEATS_EXP3], base.loc[tr_mask, "log_sales"]
Xva, yva = base.loc[va_mask, FEATS_EXP3], base.loc[va_mask, "log_sales"]
model3 = train_lgb(Xtr, ytr, Xva, yva, cat_cols=["store_nbr", "family", "city", "state", "type", "cluster"])

explainer = shap.TreeExplainer(model3)
sample_idx = np.random.RandomState(42).choice(len(Xva), size=5000, replace=False)
Xs = Xva.iloc[sample_idx]
shap_values = explainer.shap_values(Xs)

shap.summary_plot(shap_values, Xs, max_display=15)

전역 중요도(`mean|SHAP|`) 상위 피처는 다음과 같습니다 (수치는 예시 실행 기준):

| 순위 | 피처 | 의미 |
|---|---|---|
| 1 | `roll_mean_7` | 16일 전 시점 기준, 최근 7일 평균 판매량 — 압도적 1위 |
| 2 | `onpromotion` | 프로모션 여부/개수 |
| 3 | `roll_mean_60` | 장기(60일) 평균 판매량 |
| 4 | `lag_16` | 정확히 16일 전 판매량 |
| 5 | `family` | 상품군 |

**`roll_mean_7`이 압도적으로 중요합니다.** 즉 "최근 이 매장·상품이 대략 어느 정도 팔리고 있었는가"라는 수준(level) 정보가 예측의 절대다수를 차지하고, 공휴일/이벤트류는 그 위에 얹히는 미세 조정 정도의 역할이라는 뜻입니다. 이게 실험 루프 3에서 공휴일 피처의 개선폭이 작았던 이유를 설명해줍니다.

이제 **오차가 어디서 크게 나는지** 상품군별/매장타입별/요일별로 뜯어보겠습니다.

In [ ]:
pred3 = np.expm1(model3.predict(Xva, num_iteration=model3.best_iteration))
err = base.loc[va_mask, ["family", "store_nbr", "type", "dayofweek"]].copy()
err["y"] = base.loc[va_mask, "sales"].values
err["pred"] = np.clip(pred3, 0, None)
err["sq_log_err"] = (np.log1p(err.y) - np.log1p(err.pred)) ** 2

fam_rmsle = err.groupby("family").sq_log_err.mean().pow(0.5).sort_values(ascending=False)
print("상품군별 RMSLE (오차 큰 순):")
print(fam_rmsle.head(8))
print("\n상품군별 RMSLE (오차 작은 순):")
print(fam_rmsle.tail(5))

`SCHOOL AND OFFICE SUPPLIES`(학용품), `LINGERIE`, `GROCERY II` 같은 상품군의 RMSLE가 유독 높습니다. 앞서 EDA에서 봤듯 학용품류는 0 판매 비율이 74%에 달하는 간헐적 수요 상품이라, 어쩌다 한 번 몰아서 팔릴 때(개학 시즌 등) 예측이 크게 빗나가는 것으로 보입니다.

### 다음 가설
- **가설 5**: 학용품처럼 "연중 특정 시기에만 몰리는" 상품군은 최근 몇 주 패턴(`roll_mean`)만으로는 못 잡는다. **작년 같은 시기(1년 전 lag)**, **상품군×월 평균**을 추가하면 이런 계절 상품의 오차를 줄일 수 있을 것이다.
- **가설 6**: `roll_mean_7`이 저렇게 중요하다면, 최근 추세(상승/하강)를 나타내는 **모멘텀 피처**(단기평균 − 장기평균)도 도움이 될 것이다.
- **가설 7**: `store_nbr`과 `family`를 따로 넣는 것보다 **매장×상품군 조합**을 하나의 카테고리로 합치면 트리가 조합별 기본 수준을 더 빨리 찾을 것이다.

세 가지 모두 실험해보겠습니다.

## 9. 실험 루프 5, 6, 7 — SHAP 인사이트 검증 (⚠️ 모두 기각)

In [ ]:
def add_yearly_lag(df):
    df = df.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)
    df["lag_364"] = df.groupby(["store_nbr", "family"])["log_sales"].shift(364)
    return df

base = add_yearly_lag(base)

# 주의: add_safe_lag_features가 base를 (store_nbr, family, date) 기준으로 재정렬했으므로,
# 3절에서 만든 train_mask/val_mask는 더 이상 base의 행 순서와 맞지 않습니다.
# fit_cutoff는 train_mask에 의존하지 않고 base의 날짜만으로 다시 계산합니다.
fit_cutoff = base["date"].max() - pd.Timedelta(days=VAL_DAYS - 1)
fit_rows = base[base.date < fit_cutoff]
fam_month = fit_rows.groupby(["family", "month"], observed=True)["log_sales"].mean()
fam_month.name = "fam_month_mean"
base = base.merge(fam_month.reset_index(), on=["family", "month"], how="left")
base["fam_month_mean"] = base["fam_month_mean"].fillna(fit_rows["log_sales"].mean())

_ = run_experiment("exp5_yearly_season", base, CAL + ID + LAGS + PROMO + STORE_META + HOLIDAY + ["lag_364", "fam_month_mean"],
                    cat_cols=["store_nbr", "family", "city", "state", "type", "cluster"],
                    note="1년 전 lag + 상품군×월 평균(계절지수) 추가 — 학용품 등 계절 상품 개선 기대")

In [ ]:
base["store_family"] = base["store_nbr"].astype(str) + "_" + base["family"].astype(str)
base["store_family"] = base["store_family"].astype("category")

_ = run_experiment("exp6_store_family_id", base, CAL + ["store_family"] + LAGS + PROMO + STORE_META + HOLIDAY,
                    cat_cols=["store_family", "city", "state", "type", "cluster"],
                    note="store_nbr+family를 하나의 결합 카테고리로 대체")

In [ ]:
base["momentum_diff"] = base["roll_mean_7"] - base["roll_mean_30"]
base["momentum_ratio"] = base["roll_mean_7"] / (base["roll_mean_30"] + 1e-3)
MOMENTUM = ["momentum_diff", "momentum_ratio"]

_ = run_experiment("exp7_momentum", base, CAL + ID + LAGS + PROMO + STORE_META + HOLIDAY + MOMENTUM,
                    cat_cols=["store_nbr", "family", "city", "state", "type", "cluster"],
                    note="단기/장기 평균 차이(모멘텀) 단독 추가")

결과 요약:

| 실험 | RMSLE | 판정 |
|---|---|---|
| exp3 (기준) | 0.3929 | - |
| exp5_yearly_season | 0.3941 | ❌ 악화 |
| exp6_store_family_id | 0.3982 | ❌ 큰 폭 악화 |
| exp7_momentum | 0.3931 | ▬ 거의 변화 없음 |

세 가설 모두 기대와 달리 **개선에 실패**했습니다. 왜 그럴까요?

- **1년 전 lag / 상품군×월 평균 (가설 5)**: `lag_364`는 초반 1년치 데이터에는 아예 결측이라 학습 데이터가 실질적으로 줄어드는 효과가 있고, `fam_month_mean`은 상품군 단위로만 집계해서 매장별 차이를 뭉개버려 오히려 정보를 흐리게 만든 것으로 보입니다.
- **결합 카테고리 (가설 6)**: `store_nbr`(54종) × `family`(33종)를 합치면 카테고리 수가 1,782개로 늘어나 각 범주에 배정되는 데이터가 희소해지고, LightGBM 입장에서도 두 개의 작은 범주형을 따로 쓰는 것보다 분기를 짜기 더 어려워진 것으로 보입니다. **트리 모델은 이미 두 범주형을 조합한 분기를 자동으로 만들 수 있기 때문에, 억지로 합쳐줄 필요가 없었던 것**입니다.
- **모멘텀 (가설 7)**: `roll_mean_7`과 `roll_mean_30`의 차이/비율은 이미 두 원본 피처가 모델에 있으니 트리가 필요하면 스스로 계산해낼 수 있는 정보라서, 새로 추가해도 얻는 게 거의 없었습니다.

**교훈**: SHAP에서 어떤 피처가 중요하다고 해서, 그 피처를 "변형"한 파생 피처가 항상 도움이 되는 건 아닙니다. 트리 기반 모델은 이미 존재하는 피처들의 조합/차이를 스스로 학습할 수 있는 경우가 많습니다.

## 10. 실험 루프 8, 9 — 모델 자체를 튜닝해보자

피처를 더 추가하는 게 한계에 부딪혔으니, 이번엔 **모델의 복잡도**를 조정하는 실험을 해보겠습니다. 데이터가 300만 행이나 되니, 상반된 두 가설을 동시에 검증해봅니다.

- **가설 8 (정규화)**: 검증 기간(2017년 8월)은 학습 기간과 분포가 조금 다를 수 있으니(트렌드가 계속 바뀜), 모델을 더 단순하게 만들면(잎 개수 축소, L1/L2 정규화) 과적합이 줄어 오히려 더 잘 일반화될 것이다.
- **가설 9 (반대 가설)**: 데이터가 워낙 크기 때문에, 오히려 트리를 더 깊고 복잡하게 만들어야 매장×상품군별 미세한 패턴까지 잡아낼 수 있을 것이다.

In [ ]:
FEATS_BEST = CAL + ID + LAGS + PROMO + STORE_META + HOLIDAY
CAT_BEST = ["store_nbr", "family", "city", "state", "type", "cluster"]

_ = run_experiment("exp8_regularized", base, FEATS_BEST, CAT_BEST, num_round=4000,
    params=dict(num_leaves=127, min_data_in_leaf=100, lambda_l1=0.5, lambda_l2=0.5, learning_rate=0.04),
    note="정규화 강화 (num_leaves↓, min_data_in_leaf↑, L1/L2 추가)")

_ = run_experiment("exp9_deep", base, FEATS_BEST, CAT_BEST, num_round=4000,
    params=dict(num_leaves=511, min_data_in_leaf=30, learning_rate=0.04, feature_fraction=0.7),
    note="더 깊은 트리 (num_leaves 255→511)")

결과: `exp8_regularized` = **0.3923** (소폭 개선), `exp9_deep` = **0.3910** (더 큰 개선). **가설 9가 맞았습니다** — 데이터가 크고 매장×상품군 조합이 다양해서, 오히려 트리를 더 정교하게 키우는 쪽이 이득이었습니다.

여기서 멈추지 않고, 가장 좋았던 세팅(`exp9`)에 **가설 7에서 단독으론 효과 없었던 모멘텀 피처를 다시 결합**해봅니다 — 트리가 더 깊어지면 이전엔 별 도움 안 됐던 파생 피처도 다르게 쓰일 수 있기 때문입니다. 그리고 "더 깊게 가면 계속 좋아질까?"도 확인해봅니다.

In [ ]:
DEEP_PARAMS = dict(num_leaves=511, min_data_in_leaf=30, learning_rate=0.04, feature_fraction=0.7)

_ = run_experiment("exp10_deep_momentum", base, FEATS_BEST + MOMENTUM, CAT_BEST, num_round=4000,
    params=DEEP_PARAMS, note="exp9 세팅 + 모멘텀 피처 재결합")

_ = run_experiment("exp11_even_deeper", base, FEATS_BEST, CAT_BEST, num_round=4000,
    params=dict(num_leaves=767, min_data_in_leaf=20, learning_rate=0.035, feature_fraction=0.7, bagging_fraction=0.75),
    note="한 단계 더 깊게 (num_leaves 511→767)")

- `exp10_deep_momentum` = **0.3905** — 지금까지 중 최고! 트리가 깊어지니 모멘텀 피처가 실제로 도움이 됐습니다.
- `exp11_even_deeper` = **0.3916** — 오히려 다시 나빠졌습니다. `num_leaves=511` 부근이 최적점이고, 그보다 더 깊게 가면 과적합이 시작된다는 뜻입니다.

마지막으로, 가장 좋은 세팅(`exp10`)에서 **학습률만 낮춰 더 정밀하게 수렴**시켜 보겠습니다 (학습률을 낮추는 대신 라운드 수를 늘려 촘촘하게 최적점을 찾는, 자주 쓰이는 마무리 튜닝입니다).

In [ ]:
FEATS_FINAL = FEATS_BEST + MOMENTUM
FINAL_PARAMS = dict(num_leaves=511, min_data_in_leaf=30, learning_rate=0.025, feature_fraction=0.7, bagging_fraction=0.85)

model_final, score_final = run_experiment("exp12_final", base, FEATS_FINAL, CAT_BEST, num_round=5000,
    params=FINAL_PARAMS, note="exp10 세팅에서 learning_rate만 낮춰 정밀 수렴 (최종 후보)")

pd.DataFrame(results).sort_values("RMSLE").reset_index(drop=True)

**최종 RMSLE ≈ 0.389** 로, 베이스라인(0.464) 대비 **약 16% 개선**했습니다. 전체 실험을 정리하면:

| # | 실험 | RMSLE | 가설 판정 |
|---|---|---|---|
| 0 | baseline | 0.4642 | 기준선 |
| 1 | + lag/rolling | 0.4618 | ✅ (미미) |
| 2 | + promo/store meta | 0.3949 | ✅✅✅ 가장 큰 개선 |
| 3 | + holiday | 0.3929 | ✅ (소폭) |
| 4 | + oil/payday | 0.3944 | ❌ 기각 |
| 5 | + yearly season | 0.3941 | ❌ 기각 |
| 6 | + store×family id | 0.3982 | ❌ 기각 |
| 7 | + momentum(단독) | 0.3931 | ▬ 효과 없음 |
| 8 | 정규화 강화 | 0.3923 | ✅ (소폭) |
| 9 | 깊은 트리 | 0.3910 | ✅ |
| 10 | 깊은 트리 + momentum | 0.3905 | ✅ 최고 갱신 |
| 11 | 더 깊은 트리 | 0.3916 | ❌ 과적합 시작 |
| **12** | **10 + 정밀 수렴(최종)** | **0.3891** | ✅ **최종 채택** |

가설-실험-결과 루프를 12회(요구된 5회를 훌쩍 넘겨) 반복했고, 그중 절반 가까이는 기각되었습니다. **"시도했지만 안 됐다"는 결과도 왜 안 됐는지 설명할 수 있으면 그 자체로 훌륭한 분석**입니다.

## 11. 최종 모델 SHAP 재검증

10절에서 학습한 최종 모델(`model_final`, exp12_final 설정)을 그대로 다시 사용해 SHAP을 뜯어보고, 처음(8절) 모델과 비교해 어떤 피처의 역할이 달라졌는지 확인합니다. (모델을 새로 학습하지 않고 재사용합니다.)

In [ ]:
tr_mask, va_mask = time_split(base)
Xva, yva = base.loc[va_mask, FEATS_FINAL], base.loc[va_mask, "log_sales"]

explainer_f = shap.TreeExplainer(model_final)
sample_idx = np.random.RandomState(42).choice(len(Xva), size=3000, replace=False)
Xs_f = Xva.iloc[sample_idx]
shap_values_f = explainer_f.shap_values(Xs_f)

shap.summary_plot(shap_values_f, Xs_f, max_display=15)

실제로 뜯어보면 `roll_mean_7`(mean|SHAP| ≈ 0.95)이 2위인 `onpromotion`(≈ 0.28)을 3배 넘게 앞서며 여전히 압도적 1위이고, 그 뒤를 `roll_mean_60`, `family`, `lag_16`, `lag_21`, `roll_mean_14` 순으로 lag/rolling 계열 피처와 `onpromotion`, `family`가 상위권을 채우고 있습니다. 8절(exp3 기준) 분석과 순위가 거의 비슷하게 유지된 걸 확인할 수 있는데, 이는 트리를 더 깊게 만들고 모멘텀 피처를 추가해도 **"최근 판매 수준"이라는 핵심 신호의 중요도 자체는 흔들리지 않는다**는 뜻입니다. 반면 `oil_price`, `is_payday`처럼 기각했던 피처들은 애초에 최종 피처셋에서 제외했으니 등장하지 않습니다. 이렇게 **가설 검증 → 채택/기각 → SHAP 재확인**을 반복하며 피처셋을 다듬어가는 것이 실전 캐글/실무 모델링의 전형적인 흐름입니다.

오차 분석도 다시 확인해봅니다.

In [ ]:
pred_f = np.expm1(model_final.predict(Xva, num_iteration=model_final.best_iteration))
err_f = base.loc[va_mask, ["family", "type"]].copy()
err_f["y"] = base.loc[va_mask, "sales"].values
err_f["pred"] = np.clip(pred_f, 0, None)
err_f["sq_log_err"] = (np.log1p(err_f.y) - np.log1p(err_f.pred)) ** 2

print("상품군별 RMSLE (여전히 오차가 큰 상품군):")
print(err_f.groupby("family").sq_log_err.mean().pow(0.5).sort_values(ascending=False).head(6))

`SCHOOL AND OFFICE SUPPLIES`류의 오차는 여전히 큽니다. 이는 근본적으로 **간헐적 수요(intermittent demand)** 문제라서, 지금까지 쓴 "평균 기반" 피처들로는 한계가 있다는 뜻입니다 (12절 회고에서 개선 방향을 정리합니다).

## 12. 최종 제출 파일 만들기

검증에서 확인한 최적 하이퍼파라미터와 피처셋으로, 이번엔 **검증 기간까지 포함한 전체 train 데이터**로 다시 학습한 뒤 실제 `test.csv`(2017-08-16~31) 기간을 예측합니다. (검증 때 찾은 `best_iteration`만큼만 다시 학습해서, 정보를 낭비하지 않으면서도 과적합을 피합니다.)

In [ ]:
test_fe = add_calendar_features(test.copy())
test_fe["family"] = test_fe["family"].astype("category")
test_fe["store_nbr"] = test_fe["store_nbr"].astype("category")
test_fe["log_sales"] = np.nan
test_fe["sales"] = np.nan

full = pd.concat([base[["id","date","store_nbr","family","sales","onpromotion","log_sales"] + CAL], test_fe[["id","date","store_nbr","family","sales","onpromotion","log_sales"] + CAL]],
                  ignore_index=True, sort=False)
full = add_safe_lag_features(full)  # 내부에서 (store_nbr, family, date) 기준으로 정렬 후 lag/rolling 계산
full = full.merge(stores[["store_nbr","city","state","type","cluster"]], on="store_nbr", how="left")
for c in ["city","state","type"]:
    full[c] = full[c].astype("category")
full["cluster"] = full["cluster"].astype("category")
full = merge_holidays(full, holidays)
full["momentum_diff"] = full["roll_mean_7"] - full["roll_mean_30"]
full["momentum_ratio"] = full["roll_mean_7"] / (full["roll_mean_30"] + 1e-3)

is_test = full["id"] >= test["id"].min()
train_all = full[~is_test]
test_all = full[is_test]

dtrain_full = lgb.Dataset(train_all[FEATS_FINAL], label=train_all["log_sales"], categorical_feature=CAT_BEST, free_raw_data=False)
final_full_model = lgb.train({**dict(objective="regression", metric="rmse", verbose=-1, seed=42), **FINAL_PARAMS},
                              dtrain_full, num_boost_round=model_final.best_iteration)

test_pred = np.expm1(final_full_model.predict(test_all[FEATS_FINAL]))
test_pred = np.clip(test_pred, 0, None)

submission = test_all[["id"]].copy()
submission["sales"] = test_pred
submission = submission.sort_values("id")
submission.to_csv("submission.csv", index=False)
submission.head()

`submission.csv`가 만들어졌습니다. 이 파일을 캐글 대회 페이지에 제출하면 **Public/Private Leaderboard 점수**를 받아볼 수 있습니다 (이 노트북에서는 실제 정답이 없는 test 기간이라, 우리가 확인한 0.389는 어디까지나 우리가 직접 떼어놓은 검증셋 기준입니다).

## 13. 회고 — 무엇을 배웠나

### 잘 먹힌 것
1. **프로모션·매장 특성**: 가장 큰 개선을 가져온 피처. EDA에서 뚜렷하게 보였던 신호는 실제로도 강력했다.
2. **Safe lag / rolling mean**: `roll_mean_7`이 SHAP 기준 압도적 1위 — "최근에 얼마나 팔렸는가"가 예측의 근간.
3. **공휴일(대체휴일 처리 포함)**: 폭은 작지만 New Year 급락, 지진 같은 이상치성 이벤트를 잡아준 유의미한 개선.
4. **트리를 더 깊게(num_leaves 255→511)**: 300만 행, 54개 매장×33개 상품군의 다양한 패턴을 담기엔 기본 설정이 오히려 얕았다.

### 안 먹힌 것 (그리고 왜)
1. **유가(oil), 월급일(payday)**: `day` 피처와 정보가 겹치거나(payday), 학습/검증 기간의 국면 차이로 잘못된 외삽을 유도(oil).
2. **1년 전 lag, 상품군×월 평균**: 결측이 많고 매장별 차이를 뭉개서 오히려 노이즈로 작용.
3. **store×family 결합 카테고리**: 트리 모델은 이미 두 범주형의 조합을 스스로 학습할 수 있어서, 억지로 합치니 카테고리만 희소해지고 손해.
4. **모멘텀 피처(단독)**: 원본 피처들의 단순 변형이라 얕은 트리에서는 정보 가치가 거의 없었음(단, 깊은 트리와는 결합해서 도움이 됨 — 모델 복잡도에 따라 피처의 가치가 달라질 수 있다는 교훈).

### 시간이 더 있다면 시도해볼 것
- **재귀적(recursive) 다단계 예측**: 지금은 모든 시점에 최소 16일 lag만 써서 보수적으로 예측했다. 1일치씩 예측하고 그 예측값을 다음날의 lag로 재사용하면 단기 lag의 힘을 더 살릴 수 있다 (다만 예측 오차가 누적되는 위험도 있다).
- **상품군별 개별 모델**: `family`마다 판매 특성이 워낙 달라서(마진 큰 GROCERY I vs 간헐적인 BOOKS), 공용 모델 하나보다 상품군 그룹별로 모델을 나누면 더 좋아질 가능성이 있다.
- **캐글 공식 지표(NWRMSLE)**: 실제 대회는 신선식품(perishable)에 1.25배 가중치를 주는 가중 RMSLE를 쓴다. 이 그대로 학습 목적함수(`sample_weight`)에 반영하면 리더보드 점수에 더 최적화할 수 있다.
- **앙상블**: LightGBM 외에 CatBoost, 통계적 시계열 모델(Prophet, 지수평활)을 함께 써서 평균/스태킹하면 보통 추가 개선이 있다.
- **간헐적 수요 전용 기법**: `BOOKS`, `BABY CARE`처럼 0이 90%를 넘는 상품군은 일반 회귀보다 Croston's method 같은 간헐적 수요 전용 기법이나, 0/양수를 분리 예측하는 2단계 모델이 더 적합할 수 있다.